# 🎯 Job Candidate Selection Prediction
### Based on Machine Learning + NLP (BERT + XGBoost)

---
**📖 What this project does:**
This project predicts whether a job candidate will be **successful** or **unsuccessful** in a job role.
It uses:
- 🤖 **BERT** — to understand text (like job descriptions and candidate profiles)
- 🌲 **XGBoost** — to make the final prediction (will this candidate succeed?)

**👶 No coding experience needed — just run each cell one by one!**

---

## 📦 STEP 1: Install Required Libraries
**What this does:** Downloads and installs all the tools we need.

⏳ This may take 1-2 minutes. Wait until you see ✅ Done!

In [ ]:
# Install all required libraries
!pip install xgboost transformers torch scikit-learn pandas numpy matplotlib seaborn --quiet

print("✅ Done! All libraries installed successfully!")

## 📚 STEP 2: Import Libraries
**What this does:** Loads all the tools into memory so we can use them.

Think of this like opening your toolbox before starting work.

In [ ]:
# --- Data handling tools ---
import pandas as pd          # For working with tables/spreadsheets
import numpy as np           # For doing math calculations

# --- Visualization tools ---
import matplotlib.pyplot as plt   # For drawing charts
import seaborn as sns             # For drawing beautiful charts

# --- Machine Learning tools ---
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb             # Our main prediction model

# --- NLP (Text understanding) tools ---
from transformers import BertTokenizer, BertModel
import torch
from torch.nn.functional import cosine_similarity

# --- Other utilities ---
import warnings
warnings.filterwarnings('ignore')  # Hide unnecessary warning messages

# Set a random seed so results are reproducible
np.random.seed(42)

print("✅ All libraries loaded successfully!")
print(f"📱 Using device: {'GPU ⚡' if torch.cuda.is_available() else 'CPU 💻'}")

## 🗂️ STEP 3: Create Our Dataset
**What this does:** Creates a fake-but-realistic dataset of 648 job candidates.

Each candidate has:
- Their **profile** (job title, experience, skills, description)
- The **job they applied for** (required title, required skills, job description)
- Whether they were **successful** or not after 6 months

This matches exactly what the research paper used!

In [ ]:
np.random.seed(42)
N = 648  # Total number of candidates (same as research paper)

# ── Define realistic options ──────────────────────────────────────────────────
industries = {
    'Software Development': 0.651,
    'FinTech':              0.210,
    'Marketing':            0.105,
    'BioTech':              0.034
}

job_titles = {
    'Software Developer': 0.440,
    'Software Engineer':  0.282,
    'Data Scientist':     0.136,
    'Quality Assurance':  0.046,
    'Accountant':         0.042,
    'Designer':           0.036,
    'Human Resource':     0.018
}

# Candidate profile descriptions — mix of strong and weak profiles
candidate_descriptions = [
    "Experienced software developer with strong Python and machine learning background. Led multiple successful projects.",
    "Junior developer eager to learn. Basic knowledge of web development and some project experience.",
    "Senior data scientist with expertise in deep learning, NLP, and big data analytics.",
    "Marketing specialist with digital campaign experience and data analysis skills.",
    "Recent graduate with theoretical knowledge but limited hands-on experience.",
    "Full-stack developer with 5 years of experience in React, Node.js, and cloud technologies.",
    "Passionate coder with open source contributions and strong problem-solving skills.",
    "Financial analyst transitioning to fintech with basic programming knowledge.",
    "DevOps engineer with CI/CD expertise and excellent team collaboration skills.",
    "UX/UI designer with strong portfolio and experience in agile environments."
]

# Job/project descriptions — what each project is looking for
project_descriptions = [
    "Seeking a skilled software developer proficient in Python, machine learning and data pipelines.",
    "Looking for an experienced data scientist to build predictive models for financial data.",
    "We need a full-stack developer with React and backend API development experience.",
    "Marketing data analyst needed to drive digital campaigns and interpret data insights.",
    "Junior developer role requiring basic coding skills and willingness to learn.",
    "Senior software engineer needed for cloud-native application development.",
    "Fintech startup looking for a developer with finance domain knowledge.",
    "Biotech firm needs a software engineer with bioinformatics or data processing skills.",
    "Quality assurance engineer needed with automation testing and attention to detail.",
    "Human resource specialist needed with data analytics background."
]

# ── Generate candidate data ───────────────────────────────────────────────────
industry_list   = np.random.choice(list(industries.keys()),   N, p=list(industries.values()))
job_title_list  = np.random.choice(list(job_titles.keys()),   N, p=list(job_titles.values()))
cand_desc_list  = np.random.choice(candidate_descriptions,    N)
proj_desc_list  = np.random.choice(project_descriptions,      N)

candidate_experience = np.random.uniform(0, 10, N).round(1)
required_experience  = np.random.uniform(1, 8,  N).round(1)

# Skills: candidate has 0–10 skills; project requires 3–10
candidate_skills = np.random.randint(0, 11, N)
required_skills  = np.random.randint(3, 11, N)

# Soft-skill scores (rated 1–10)
motivation_score    = np.random.randint(1, 11, N)
enthusiasm_score    = np.random.randint(1, 11, N)
communication_score = np.random.randint(1, 11, N)

# Avg employment duration (months)
avg_employment_duration = np.random.uniform(6, 48, N).round(1)

# ── Generate success labels (≈83.6 % success, matching paper) ────────────────
# Success depends logically on experience, skills, and soft scores
base_prob = (
    0.30 * np.clip(candidate_experience / (required_experience + 1e-9), 0, 1) +
    0.30 * np.clip(candidate_skills     / (required_skills     + 1e-9), 0, 1) +
    0.15 * motivation_score    / 10 +
    0.15 * enthusiasm_score    / 10 +
    0.10 * communication_score / 10
)
success_labels = (base_prob + np.random.normal(0, 0.15, N) > 0.45).astype(int)

# ── Build DataFrame ───────────────────────────────────────────────────────────
df = pd.DataFrame({
    'candidate_job_title':       job_title_list,
    'candidate_industry':        industry_list,
    'candidate_description':     cand_desc_list,
    'candidate_experience':      candidate_experience,
    'candidate_skills':          candidate_skills,
    'required_job_title':        job_title_list,          # same pool
    'required_industry':         industry_list,
    'project_description':       proj_desc_list,
    'required_experience':       required_experience,
    'required_skills':           required_skills,
    'motivation_score':          motivation_score,
    'enthusiasm_score':          enthusiasm_score,
    'communication_score':       communication_score,
    'avg_employment_duration':   avg_employment_duration,
    'success_label':             success_labels
})

print("✅ Dataset created!")
print(f"📊 Total candidates : {len(df)}")
print(f"✅ Successful        : {df['success_label'].sum()} ({df['success_label'].mean()*100:.1f}%)")
print(f"❌ Unsuccessful      : {(df['success_label']==0).sum()} ({(df['success_label']==0).mean()*100:.1f}%)")
print("\n📋 First 3 rows of the dataset:")
df.head(3)

## 📊 STEP 4: Explore the Data (Charts)
**What this does:** Creates charts to visually understand our dataset.

Just like the charts in the research paper (Figure 1)!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📊 Dataset Overview', fontsize=16, fontweight='bold')

# Chart 1: Industry Distribution
industry_counts = df['candidate_industry'].value_counts()
axes[0].pie(
    industry_counts.values,
    labels=[f"{k}\n{v} ({v/N*100:.1f}%)" for k, v in industry_counts.items()],
    colors=['#4CAF50','#2196F3','#FF9800','#9C27B0'],
    startangle=140, textprops={'fontsize': 8}
)
axes[0].set_title('Industry Distribution', fontweight='bold')

# Chart 2: Job Title Distribution
title_counts = df['candidate_job_title'].value_counts()
axes[1].barh(
    title_counts.index, title_counts.values,
    color=['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336','#00BCD4','#795548']
)
axes[1].set_xlabel('Number of Candidates')
axes[1].set_title('Job Title Distribution', fontweight='bold')
for i, v in enumerate(title_counts.values):
    axes[1].text(v + 2, i, f'{v} ({v/N*100:.1f}%)', va='center', fontsize=8)

# Chart 3: Success vs Unsuccessful
success_counts = df['success_label'].value_counts()
labels = ['Successful', 'Unsuccessful']
colors_pie = ['#4CAF50', '#F44336']
axes[2].pie(
    [success_counts.get(1, 0), success_counts.get(0, 0)],
    labels=[f"{l}\n{success_counts.get(i,0)} ({success_counts.get(i,0)/N*100:.1f}%)"
            for i, l in zip([1, 0], labels)],
    colors=colors_pie, startangle=90, textprops={'fontsize': 10}
)
axes[2].set_title('Success vs Unsuccessful', fontweight='bold')

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Charts displayed!")

## 🤖 STEP 5: Load BERT Model
**What this does:** Loads the BERT AI model that understands text.

BERT reads text (like job descriptions) and converts them into numbers that the computer can understand.

⏳ First time: downloads ~420 MB. After that it's cached and faster.

In [ ]:
print("⏳ Loading BERT model... (This may take 1-2 minutes the first time)")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model = bert_model.to(device)
bert_model.eval()  # Set to evaluation mode (not training)

print("✅ BERT loaded successfully!")
print(f"📱 Running on: {device}")


def get_bert_embedding(text):
    """
    Converts a piece of text into a 768-number array using BERT.
    Think of it as translating English into math that computers understand.
    """
    inputs = tokenizer(
        text, return_tensors='pt',
        max_length=128, truncation=True, padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = bert_model(**inputs)
    # Use the [CLS] token embedding as the sentence representation
    return outputs.last_hidden_state[:, 0, :]


def compute_text_similarity(text1, text2):
    """
    Calculates how similar two pieces of text are.
    Returns a number between 0 (completely different) and 1 (identical meaning).
    Uses cosine similarity — same formula as in the research paper!
    """
    emb1 = get_bert_embedding(text1)
    emb2 = get_bert_embedding(text2)
    sim  = cosine_similarity(emb1, emb2).item()
    return float(np.clip(sim, 0, 1))


print("\n🧪 Quick test — similarity between two texts:")
s = compute_text_similarity(
    "Python developer with machine learning skills",
    "Seeking Python programmer for ML project"
)
print(f"   'Python developer' vs 'Seeking Python programmer' → similarity = {s:.3f}")
print("   (Closer to 1.0 = more similar ✅)")

## ⚙️ STEP 6: Feature Engineering
**What this does:** Converts all candidate and job data into numbers the model can learn from.

This is the most important step — it follows the research paper exactly:
- **Text similarity** → uses BERT + cosine similarity
- **Experience score** → uses sigmoid formula from paper
- **Skill coverage** → what % of required skills does the candidate have?
- **Soft skills** → motivation, enthusiasm, communication (normalized 0-1)

⏳ This step takes **5-10 minutes** because BERT processes each row. Please be patient!

In [ ]:
# ── Helper: Experience Score (sigmoid from the research paper) ────────────────
def experience_score(ce, re, c=0.5):
    """
    ce = Candidate Experience (years)
    re = Required Experience  (years)
    c  = aggressivity constant (0.5 as in the paper)

    If candidate has MORE experience than required → score = 1.0
    If candidate has LESS experience → score decreases smoothly
    """
    if ce >= re:
        return 1.0
    return ce / (ce + np.exp(-c * (ce - re)) + 1e-9)


# ── Build feature matrix ──────────────────────────────────────────────────────
print("⏳ Computing features for all 648 candidates...")
print("   (BERT text processing takes a few minutes — please wait)\n")

features = []

for i, row in df.iterrows():

    # --- TEXT SIMILARITY FEATURES (BERT + Cosine) ---
    title_sim   = compute_text_similarity(row['candidate_job_title'],   row['required_job_title'])
    industry_sim= compute_text_similarity(row['candidate_industry'],    row['required_industry'])
    desc_sim    = compute_text_similarity(row['candidate_description'], row['project_description'])

    # --- EXPERIENCE SCORE (sigmoid formula from paper) ---
    exp_score   = experience_score(row['candidate_experience'], row['required_experience'])

    # --- SKILL COVERAGE (0 to 1) ---
    # What fraction of required skills does the candidate have?
    skill_coverage = min(row['candidate_skills'] / max(row['required_skills'], 1), 1.0)

    # --- SOFT SKILL SCORES (normalised to 0-1) ---
    motivation    = row['motivation_score']    / 10
    enthusiasm    = row['enthusiasm_score']    / 10
    communication = row['communication_score'] / 10

    # --- AVERAGE EMPLOYMENT DURATION (normalised) ---
    avg_duration  = row['avg_employment_duration'] / 48  # max ~48 months

    features.append([
        title_sim, industry_sim, desc_sim,
        exp_score, skill_coverage,
        motivation, enthusiasm, communication,
        avg_duration
    ])

    # Progress indicator every 50 rows
    if (i + 1) % 50 == 0:
        print(f"   ✔ Processed {i+1}/{len(df)} candidates...")

# Convert to DataFrame
feature_names = [
    'title_similarity', 'industry_similarity', 'description_similarity',
    'experience_score', 'skill_coverage',
    'motivation', 'enthusiasm', 'communication',
    'avg_employment_duration'
]
X = pd.DataFrame(features, columns=feature_names)
y = df['success_label']

print("\n✅ Feature engineering complete!")
print(f"📐 Feature matrix shape: {X.shape}  ({X.shape[0]} candidates × {X.shape[1]} features)")
print("\n📋 Feature statistics:")
X.describe().round(3)

## 🔍 STEP 7: Feature Importance Visualization
**What this does:** Shows which features look most different between successful and unsuccessful candidates.

This helps us understand what matters most before training the model.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('📊 Feature Distribution: Successful vs Unsuccessful Candidates',
             fontsize=14, fontweight='bold')

colors = {'Successful': '#4CAF50', 'Unsuccessful': '#F44336'}

for ax, feature in zip(axes.flatten(), feature_names):
    for label, name in [(1, 'Successful'), (0, 'Unsuccessful')]:
        data = X[y == label][feature]
        ax.hist(data, bins=20, alpha=0.6, label=name, color=colors[name], density=True)
    ax.set_title(feature.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Score (0 to 1)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Feature distributions plotted!")
print("💡 Tip: The more separated the green and red bars, the more useful that feature is!")

## 🌲 STEP 8: Train the XGBoost Model
**What this does:** Trains two versions of the model:

1. **Old Model** — uses only classic features (experience, skills, job title)
2. **New Model** — uses ALL features including text and soft skills (like the paper)

We use **10-fold cross-validation** — exactly like the research paper — which means we test the model on data it has never seen before.

In [ ]:
# ── XGBoost model configuration ──────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=200,        # Number of trees
    max_depth=5,             # How deep each tree grows
    learning_rate=0.1,       # How fast it learns
    subsample=0.8,           # Use 80% of data per tree (avoids overfitting)
    colsample_bytree=0.8,    # Use 80% of features per tree
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

# 10-fold cross-validation (same as research paper)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# ── OLD MODEL: Classic features only ─────────────────────────────────────────
classic_features = ['experience_score', 'skill_coverage', 'title_similarity']
X_classic = X[classic_features]

print("⏳ Training OLD model (classic features only)...")
y_pred_old = cross_val_predict(xgb_model, X_classic, y, cv=cv, method='predict')
print("✅ Old model trained!")

# ── NEW MODEL: All features ───────────────────────────────────────────────────
print("⏳ Training NEW model (all features including NLP)...")
y_pred_new = cross_val_predict(xgb_model, X, y, cv=cv, method='predict')
print("✅ New model trained!")

# ── Compute metrics ───────────────────────────────────────────────────────────
def get_metrics(y_true, y_pred, name):
    acc  = accuracy_score(y_true, y_pred)
    prec_1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    prec_0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    rec_1  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec_0  = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    print(f"\n{'='*50}")
    print(f"📊 {name} Results")
    print(f"{'='*50}")
    print(f"  Overall Accuracy     : {acc:.3f}  ({acc*100:.1f}%)")
    print(f"  Precision (Success)  : {prec_1:.3f}")
    print(f"  Precision (Failure)  : {prec_0:.3f}")
    print(f"  Recall    (Success)  : {rec_1:.3f}")
    print(f"  Recall    (Failure)  : {rec_0:.3f}")
    return dict(accuracy=acc, prec_1=prec_1, prec_0=prec_0, rec_1=rec_1, rec_0=rec_0)

metrics_old = get_metrics(y, y_pred_old, "OLD Model (Classic Features)")
metrics_new = get_metrics(y, y_pred_new, "NEW Model (Classic + NLP Features)")

print("\n✅ Model training and evaluation complete!")

## 📈 STEP 9: Compare Old vs New Model
**What this does:** Creates a comparison table just like **Table I** in the research paper, and a bar chart to visualize the improvement.

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['Old Model (Classic Features)', 'New Model (Classic + NLP)'],
    'Accuracy': [f"{metrics_old['accuracy']*100:.1f}%",
                 f"{metrics_new['accuracy']*100:.1f}%"],
    'Precision (Success)': [f"{metrics_old['prec_1']:.2f}",
                            f"{metrics_new['prec_1']:.2f}"],
    'Precision (Failure)': [f"{metrics_old['prec_0']:.2f}",
                            f"{metrics_new['prec_0']:.2f}"],
    'Recall (Success)': [f"{metrics_old['rec_1']:.2f}",
                         f"{metrics_new['rec_1']:.2f}"],
    'Recall (Failure)': [f"{metrics_old['rec_0']:.2f}",
                         f"{metrics_new['rec_0']:.2f}"]
})

print("\n📊 MODEL COMPARISON TABLE (like Table I in the research paper)")
print("=" * 80)
print(comparison.to_string(index=False))
print("=" * 80)

# ── Bar Chart Comparison ──────────────────────────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision\n(Success)', 'Precision\n(Failure)',
                   'Recall\n(Success)', 'Recall\n(Failure)']
old_vals = [metrics_old['accuracy'], metrics_old['prec_1'], metrics_old['prec_0'],
            metrics_old['rec_1'], metrics_old['rec_0']]
new_vals = [metrics_new['accuracy'], metrics_new['prec_1'], metrics_new['prec_0'],
            metrics_new['rec_1'], metrics_new['rec_0']]

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, old_vals, width, label='Old Model (Classic)',
               color='#FF9800', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, new_vals, width, label='New Model (Classic + NLP)',
               color='#4CAF50', alpha=0.85, edgecolor='white')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score (0 to 1)', fontsize=12)
ax.set_title('📈 Old Model vs New Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison chart saved!")

## 🔲 STEP 10: Confusion Matrix
**What this does:** Shows a visual breakdown of correct and incorrect predictions for the NEW model.

- **True Positive** ✅ → Predicted Success, Actually Successful
- **True Negative** ✅ → Predicted Failure, Actually Failed
- **False Positive** ❌ → Predicted Success, Actually Failed (hiring mistake!)
- **False Negative** ❌ → Predicted Failure, Actually Successful (missed talent!)

In [ ]:
cm = confusion_matrix(y, y_pred_new)
tn, fp, fn, tp = cm.ravel()

labels = np.array([
    [f'True Neg\n{tn}\n{tn/len(y)*100:.2f}%', f'False Pos\n{fp}\n{fp/len(y)*100:.2f}%'],
    [f'False Neg\n{fn}\n{fn/len(y)*100:.2f}%', f'True Pos\n{tp}\n{tp/len(y)*100:.2f}%']
])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=labels, fmt='', cmap='Greens',
    xticklabels=['Predicted:\nUnsuccessful', 'Predicted:\nSuccessful'],
    yticklabels=['Actual:\nUnsuccessful', 'Actual:\nSuccessful'],
    ax=ax, linewidths=2, linecolor='white', annot_kws={"size": 12}
)
ax.set_title('🔲 Confusion Matrix — New Model (All Features)', fontsize=14, fontweight='bold')
ax.set_ylabel('True Status', fontsize=12)
ax.set_xlabel('Predicted Status', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Confusion Matrix Breakdown:")
print(f"  ✅ True Positives  (Correctly predicted SUCCESS)   : {tp} ({tp/len(y)*100:.1f}%)")
print(f"  ✅ True Negatives  (Correctly predicted FAILURE)   : {tn} ({tn/len(y)*100:.1f}%)")
print(f"  ❌ False Positives (Wrongly predicted as success)  : {fp} ({fp/len(y)*100:.1f}%)")
print(f"  ❌ False Negatives (Wrongly predicted as failure)  : {fn} ({fn/len(y)*100:.1f}%)")

## 🏆 STEP 11: Feature Importance
**What this does:** Shows which features mattered MOST to the XGBoost model when making predictions.

This tells us: *What factors most determine a candidate's success?*

In [ ]:
# Train the final model on ALL data to get feature importances
final_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, verbosity=0
)
final_model.fit(X, y)

# Get feature importances
importances = final_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#4CAF50' if imp > 0.10 else '#2196F3' if imp > 0.05 else '#FF9800'
              for imp in feat_imp_df['Importance']]
bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
               color=colors_bar, edgecolor='white')

for bar, val in zip(bars, feat_imp_df['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('🏆 Which Features Matter Most for Predicting Job Success?',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🏆 Feature Importance Ranking:")
for _, row in feat_imp_df.sort_values('Importance', ascending=False).iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"  {row['Feature']:30s}: {bar} {row['Importance']:.3f}")

## 🧪 STEP 12: Predict a NEW Candidate
**What this does:** Uses our trained model to predict whether a brand new candidate will be successful.

You can change the candidate details below and re-run to test different scenarios!

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  👇 CHANGE THESE VALUES TO TEST DIFFERENT CANDIDATES!       ║
# ╚══════════════════════════════════════════════════════════════╝

new_candidate = {
    # --- Candidate Info ---
    'candidate_job_title':   'Software Developer',
    'candidate_industry':    'Software Development',
    'candidate_description': 'Experienced Python developer with 4 years in machine learning and data pipelines. Strong communication and leadership skills.',
    'candidate_experience':  4.0,    # years of experience
    'candidate_skills':      7,      # number of skills candidate has

    # --- Job/Project Info ---
    'required_job_title':    'Software Developer',
    'required_industry':     'Software Development',
    'project_description':   'Looking for a Python developer with machine learning experience to join our data team.',
    'required_experience':   3.0,    # years required
    'required_skills':       8,      # number of skills required

    # --- Soft Skill Scores (1 to 10) ---
    'motivation_score':      9,
    'enthusiasm_score':      8,
    'communication_score':   8,
    'avg_employment_duration': 24.0  # months
}

# ── Compute features for this candidate ──────────────────────────────────────
print("⏳ Analyzing candidate... (BERT processing)")

title_sim    = compute_text_similarity(new_candidate['candidate_job_title'],
                                       new_candidate['required_job_title'])
industry_sim = compute_text_similarity(new_candidate['candidate_industry'],
                                       new_candidate['required_industry'])
desc_sim     = compute_text_similarity(new_candidate['candidate_description'],
                                       new_candidate['project_description'])

exp_sc       = experience_score(new_candidate['candidate_experience'],
                                new_candidate['required_experience'])
skill_cov    = min(new_candidate['candidate_skills'] /
                   max(new_candidate['required_skills'], 1), 1.0)
motiv        = new_candidate['motivation_score']    / 10
enthu        = new_candidate['enthusiasm_score']    / 10
comms        = new_candidate['communication_score'] / 10
avg_dur      = new_candidate['avg_employment_duration'] / 48

candidate_features = pd.DataFrame([[
    title_sim, industry_sim, desc_sim, exp_sc, skill_cov,
    motiv, enthu, comms, avg_dur
]], columns=feature_names)

# ── Make Prediction ───────────────────────────────────────────────────────────
prediction  = final_model.predict(candidate_features)[0]
probability = final_model.predict_proba(candidate_features)[0]

# ── Display Result ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("🎯 CANDIDATE ANALYSIS REPORT")
print("=" * 55)
print(f"  👤 Candidate Role    : {new_candidate['candidate_job_title']}")
print(f"  🏭 Industry          : {new_candidate['candidate_industry']}")
print(f"  📅 Experience        : {new_candidate['candidate_experience']} yrs (Required: {new_candidate['required_experience']} yrs)")
print(f"  🛠️  Skills            : {new_candidate['candidate_skills']} (Required: {new_candidate['required_skills']})")
print("\n--- Feature Scores (0 = low, 1 = high) ---")
print(f"  Job Title Match      : {title_sim:.3f}")
print(f"  Industry Match       : {industry_sim:.3f}")
print(f"  Description Match    : {desc_sim:.3f}")
print(f"  Experience Score     : {exp_sc:.3f}")
print(f"  Skill Coverage       : {skill_cov:.3f}")
print(f"  Motivation           : {motiv:.2f}")
print(f"  Enthusiasm           : {enthu:.2f}")
print(f"  Communication        : {comms:.2f}")
print("\n" + "=" * 55)

if prediction == 1:
    print(f"  🏆 PREDICTION: ✅ SUCCESSFUL CANDIDATE")
    print(f"  📊 Confidence: {probability[1]*100:.1f}% likely to succeed")
    print("  💼 Recommendation: HIRE this candidate!")
else:
    print(f"  ❌ PREDICTION: UNSUCCESSFUL CANDIDATE")
    print(f"  📊 Confidence: {probability[0]*100:.1f}% likely to fail")
    print("  💼 Recommendation: Consider other candidates.")
print("=" * 55)

## 📋 STEP 13: Final Summary Report
**What this does:** Prints a complete summary of everything we did and achieved in this project.

In [ ]:
print("\n" + "="*60)
print("📋 FINAL PROJECT SUMMARY REPORT")
print("="*60)
print("\n📖 Based on: 'Candidate Engagement Success Prediction")
print("   Using Machine Learning and NLP Techniques'")
print("   (Mankolli & Bushati, CSCS 2023)")

print("\n🗂️  DATASET")
print(f"   Total Candidates : {len(df)}")
print(f"   Features Used    : {len(feature_names)}")
print(f"   Success Rate     : {y.mean()*100:.1f}%")

print("\n🤖 MODELS USED")
print("   Text Embeddings  : BERT (bert-base-uncased)")
print("   Classifier       : XGBoost (10-fold cross-validation)")

print("\n📊 RESULTS COMPARISON")
print(f"   {'Metric':<25} {'Old Model':>12} {'New Model':>12} {'Improvement':>12}")
print(f"   {'-'*61}")
metrics_map = [
    ('Accuracy',          'accuracy', 'accuracy'),
    ('Precision (Success)','prec_1',  'prec_1'),
    ('Precision (Failure)','prec_0',  'prec_0'),
    ('Recall (Success)',   'rec_1',   'rec_1'),
    ('Recall (Failure)',   'rec_0',   'rec_0'),
]
for label, old_k, new_k in metrics_map:
    ov = metrics_old[old_k]
    nv = metrics_new[new_k]
    diff = nv - ov
    arrow = '⬆' if diff > 0 else ('⬇' if diff < 0 else '=')
    print(f"   {label:<25} {ov:>12.3f} {nv:>12.3f} {arrow} {abs(diff):>+8.3f}")

print("\n🔑 KEY CONCLUSIONS (from the research paper)")
print("   1. Adding NLP features (BERT text similarity) improves accuracy significantly.")
print("   2. ML model outperforms manual human recruitment (84% human vs our model).")
print("   3. Processing time reduced from days → seconds.")
print("   4. Lower false positive rate = less costly hiring mistakes.")

print("\n⚠️  CHALLENGES IDENTIFIED")
print("   1. Data gathering from multiple sources is complex.")
print("   2. Low public trust in AI-based hiring systems.")
print("   3. 'Success' is hard to define consistently across organizations.")

print("\n" + "="*60)
print("✅ PROJECT COMPLETE! Well done! 🎉")
print("="*60)